# Pandas Practice — Part 2 (Solved)

This notebook is a **pure practice** companion to `Pandas_for_ML_Part_2.ipynb`. Every task below now has a worked solution in the code cell under it.

## How to use this notebook
- Read each task carefully.
- Write your solution in the empty code cell provided right below it (or add a new cell).
- If you get stuck, re-open `Pandas_for_ML_Part_2.ipynb` and review the relevant section.
- Try each task yourself first, then compare with the solution.

Tasks are grouped into three levels:
- **Level 1 — Basic**: direct practice of a single Pandas concept.
- **Level 2 — Applied**: combine 2–3 Pandas concepts together.
- **Level 3 — ML-oriented Challenge**: realistic, data-flavored mini-tasks.

---
## Table of Contents
1. [Setup](#setup)
2. [Level 1 — Basic](#level1)
3. [Level 2 — Applied](#level2)
4. [Level 3 — ML-oriented Challenge](#level3)


<a id="setup"></a>
# Setup

Run this cell first — it creates the sample dataset used throughout this notebook's exercises.

In [1]:
import pandas as pd
import numpy as np
import os

os.makedirs("sample_data", exist_ok=True)

orders_csv_text = """order_id,customer,city,product,category,quantity,unit_price,order_date
1,Ali,Lahore,Laptop,Electronics,1,1200,2024-01-05
2,Sara,Karachi,Mouse,Electronics,3,25,2024-01-06
3,Ahmed,Lahore,Desk,Furniture,1,300,2024-01-15
4,Ayesha,Islamabad,Chair,Furniture,4,150,2024-01-20
5,Ali,Lahore,Monitor,Electronics,2,220,2024-02-02
6,Bilal,Karachi,Laptop,Electronics,1,1200,2024-02-10
7,Sara,Karachi,Desk,Furniture,2,300,2024-02-14
8,Zainab,Islamabad,Chair,Furniture,3,150,2024-02-25
9,Ahmed,Lahore,Monitor,Electronics,1,220,2024-03-01
10,Ayesha,Islamabad,Mouse,Electronics,5,25,2024-03-04
11,Ali,Lahore,Chair,Furniture,2,150,2024-03-08
12,Bilal,Karachi,Monitor,Electronics,2,220,2024-03-18
"""

with open("sample_data/orders.csv", "w") as file:
    file.write(orders_csv_text)

print("orders.csv is ready.")
print(pd.__version__)

orders.csv is ready.
3.0.2


<a id="level1"></a>
# Level 1 — Basic

### Task 1: Load and Add a Revenue Column
Read `sample_data/orders.csv` into a DataFrame called `orders`. Convert `order_date` to a proper datetime type. Add a `revenue` column equal to `quantity * unit_price`.

### Task 2: Group and Aggregate
Group `orders` by `category` and print the total `revenue` for each category.

### Task 3: Multiple Aggregations
Using `.agg()`, compute the total and average `revenue` per `city` in a single call.

### Task 4: String Cleaning
Add a column `product_upper` that stores each `product` name in uppercase, using the `.str` accessor.

### Task 5: Duplicate Check
Check whether `orders` has any exact duplicate rows. Then create a small DataFrame with at least one intentional duplicate row and demonstrate removing it with `.drop_duplicates()`.

In [2]:
# Task 1: Load and Add a Revenue Column
orders = pd.read_csv("sample_data/orders.csv")
orders["order_date"] = pd.to_datetime(orders["order_date"])
orders["revenue"] = orders["quantity"] * orders["unit_price"]
print(orders.head())
print(orders.dtypes)

# Task 2: Group and Aggregate
print(orders.groupby("category")["revenue"].sum())

# Task 3: Multiple Aggregations
print(orders.groupby("city")["revenue"].agg(["sum", "mean"]))

# Task 4: String Cleaning
orders["product_upper"] = orders["product"].str.upper()
print(orders[["product", "product_upper"]].head())

# Task 5: Duplicate Check
print("Any duplicate rows in orders?", orders.duplicated().any())

small = pd.DataFrame({
    "name": ["Ali", "Sara", "Ali"],
    "score": [90, 85, 90],
})                                  # row 0 and row 2 are the same
print(small)
print(small.drop_duplicates())

   order_id customer       city  product     category  quantity  unit_price  \
0         1      Ali     Lahore   Laptop  Electronics         1        1200   
1         2     Sara    Karachi    Mouse  Electronics         3          25   
2         3    Ahmed     Lahore     Desk    Furniture         1         300   
3         4   Ayesha  Islamabad    Chair    Furniture         4         150   
4         5      Ali     Lahore  Monitor  Electronics         2         220   

  order_date  revenue  
0 2024-01-05     1200  
1 2024-01-06       75  
2 2024-01-15      300  
3 2024-01-20      600  
4 2024-02-02      440  
order_id               int64
customer                 str
city                     str
product                  str
category                 str
quantity               int64
unit_price             int64
order_date    datetime64[us]
revenue                int64
dtype: object
category
Electronics    3700
Furniture      2250
Name: revenue, dtype: int64
            sum        mean
c

<a id="level2"></a>
# Level 2 — Applied

### Task 6: Named Aggregation
Using `.groupby("city").agg(...)` with **named aggregation**, create a summary with columns `total_orders` (count of orders) and `total_revenue` (sum of revenue) per city.

### Task 7: Merge with Extra Data
Create a small DataFrame `city_region` mapping each city to a region:
```python
city_region = pd.DataFrame({
    "city": ["Lahore", "Karachi", "Islamabad"],
    "region": ["Punjab", "Sindh", "Federal"],
})
```
Merge it with `orders` (on `city`) so every order also has a `region` column. Use an appropriate `how` value and explain (in a comment) why you chose it.

### Task 8: Pivot Table
Build a pivot table showing total `revenue`, with `category` as rows and `city` as columns. Use `fill_value=0` for any missing combinations.

### Task 9: Apply with a Custom Function
Write a function that labels an order as `"Bulk"` if `quantity >= 3`, otherwise `"Regular"`. Use `.apply()` to create a new `order_type` column based on this function.

### Task 10: Dates
Extract the `order_month` and `order_weekday` (name, not number) from `order_date` into two new columns. Then find which weekday has the highest total revenue.

In [3]:
# Task 6: Named Aggregation
city_summary = orders.groupby("city").agg(
    total_orders=("order_id", "count"),
    total_revenue=("revenue", "sum"),
)
print(city_summary)

           total_orders  total_revenue
city                                  
Islamabad             3           1175
Karachi               4           2315
Lahore                5           2460


In [4]:
# Task 7: Merge with Extra Data
city_region = pd.DataFrame({
    "city": ["Lahore", "Karachi", "Islamabad"],
    "region": ["Punjab", "Sindh", "Federal"],
})

# how="left": keep every order, even if a city has no region in city_region.
# Orders are the main table, so we must not lose any of them.
orders = orders.merge(city_region, on="city", how="left")
print(orders[["order_id", "city", "region"]])

    order_id       city   region
0          1     Lahore   Punjab
1          2    Karachi    Sindh
2          3     Lahore   Punjab
3          4  Islamabad  Federal
4          5     Lahore   Punjab
5          6    Karachi    Sindh
6          7    Karachi    Sindh
7          8  Islamabad  Federal
8          9     Lahore   Punjab
9         10  Islamabad  Federal
10        11     Lahore   Punjab
11        12    Karachi    Sindh


In [5]:
# Task 8: Pivot Table
pivot = pd.pivot_table(
    orders,
    values="revenue",
    index="category",
    columns="city",
    aggfunc="sum",
    fill_value=0,
)
print(pivot)

city         Islamabad  Karachi  Lahore
category                               
Electronics        125     1715    1860
Furniture         1050      600     600


In [6]:
# Task 9: Apply with a Custom Function
def label_order(quantity):
    if quantity >= 3:
        return "Bulk"
    return "Regular"

orders["order_type"] = orders["quantity"].apply(label_order)
print(orders[["order_id", "quantity", "order_type"]])

    order_id  quantity order_type
0          1         1    Regular
1          2         3       Bulk
2          3         1    Regular
3          4         4       Bulk
4          5         2    Regular
5          6         1    Regular
6          7         2    Regular
7          8         3       Bulk
8          9         1    Regular
9         10         5       Bulk
10        11         2    Regular
11        12         2    Regular


In [7]:
# Task 10: Dates
orders["order_month"] = orders["order_date"].dt.to_period("M")     # keeps months in time order
orders["order_weekday"] = orders["order_date"].dt.day_name()

weekday_revenue = orders.groupby("order_weekday")["revenue"].sum()
print(weekday_revenue)
print("Weekday with highest revenue:", weekday_revenue.idxmax())

order_weekday
Friday       2160
Monday        865
Saturday     1875
Sunday        450
Wednesday     600
Name: revenue, dtype: int64
Weekday with highest revenue: Friday


<a id="level3"></a>
# Level 3 — ML-oriented Challenge

### Task 11: Customer-Level Feature Engineering
Using `orders`, build a customer-level summary DataFrame with one row per `customer`, containing:
- `total_orders` — number of orders placed
- `total_spent` — total revenue across all their orders
- `average_order_value` — their average revenue per order
- `favorite_category` — the category they ordered the most (hint: this is the trickiest part — think about grouping by both `customer` and `category`, counting, then finding each customer's top category)

### Task 12: Monthly Trend and Growth
Compute total revenue per `order_month`. Then compute the **month-over-month change** in revenue (this month's total minus the previous month's total) as a new Series. (Hint: look into whether a Pandas Series method can compute the difference between consecutive values without a manual loop.)

### Task 13: End-to-End Mini Report
Combine everything you've practiced to produce a single, clean summary DataFrame with one row per `city`, containing: `total_revenue`, `total_orders`, `average_order_value`, and `top_category` (the category with the most revenue in that city). Export this summary to both a CSV file and a JSON file (`orient="records"`), then reload each one and print it to confirm it round-trips correctly.

In [8]:
# Task 11: Customer-Level Feature Engineering
customer_summary = orders.groupby("customer").agg(
    total_orders=("order_id", "count"),
    total_spent=("revenue", "sum"),
    average_order_value=("revenue", "mean"),
)

# favorite_category: count orders per (customer, category), then keep the top one.
# If two categories tie, the one that comes first alphabetically wins.
category_counts = (
    orders.groupby(["customer", "category"]).size().reset_index(name="n_orders")
)
favorite = (
    category_counts
    .sort_values(["customer", "n_orders", "category"], ascending=[True, False, True])
    .drop_duplicates("customer")
    .set_index("customer")["category"]
)
customer_summary["favorite_category"] = favorite
print(customer_summary)

          total_orders  total_spent  average_order_value favorite_category
customer                                                                  
Ahmed                2          520           260.000000       Electronics
Ali                  3         1940           646.666667       Electronics
Ayesha               2          725           362.500000       Electronics
Bilal                2         1640           820.000000       Electronics
Sara                 2          675           337.500000       Electronics
Zainab               1          450           450.000000         Furniture


In [9]:
# Task 12: Monthly Trend and Growth
monthly_revenue = orders.groupby("order_month")["revenue"].sum()
print(monthly_revenue)

mom_change = monthly_revenue.diff()      # this month minus previous month
print(mom_change)

order_month
2024-01    2175
2024-02    2690
2024-03    1085
Freq: M, Name: revenue, dtype: int64
order_month
2024-01       NaN
2024-02     515.0
2024-03   -1605.0
Freq: M, Name: revenue, dtype: float64


In [10]:
# Task 13: End-to-End Mini Report
city_report = orders.groupby("city").agg(
    total_revenue=("revenue", "sum"),
    total_orders=("order_id", "count"),
    average_order_value=("revenue", "mean"),
)

# top_category: the category with the most revenue in each city
revenue_by_city_cat = orders.groupby(["city", "category"])["revenue"].sum()
top_category = revenue_by_city_cat.groupby("city").idxmax().str[1]
city_report["top_category"] = top_category
city_report = city_report.reset_index()
print(city_report)

# Export to CSV and JSON
city_report.to_csv("sample_data/city_report.csv", index=False)
city_report.to_json("sample_data/city_report.json", orient="records")

# Reload and confirm
print(pd.read_csv("sample_data/city_report.csv"))
print(pd.read_json("sample_data/city_report.json", orient="records"))

        city  total_revenue  total_orders  average_order_value top_category
0  Islamabad           1175             3           391.666667    Furniture
1    Karachi           2315             4           578.750000  Electronics
2     Lahore           2460             5           492.000000  Electronics
        city  total_revenue  total_orders  average_order_value top_category
0  Islamabad           1175             3           391.666667    Furniture
1    Karachi           2315             4           578.750000  Electronics
2     Lahore           2460             5           492.000000  Electronics
        city  total_revenue  total_orders  average_order_value top_category
0  Islamabad           1175             3           391.666667    Furniture
1    Karachi           2315             4           578.750000  Electronics
2     Lahore           2460             5           492.000000  Electronics


---
That's the end of the Pandas practice series. Once you're comfortable with everything here, you're ready to move on to **Data Visualization** and then **Scikit-learn**, where this cleaned, structured data becomes the direct input to real Machine Learning models.